# C03 — Embeddings semânticos e recuperação de informação

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

Este notebook corresponde à terceira etapa do projeto da disciplina "Sistemas Cognitivos com LLMs". O objetivo é
indexar o corpus das aulas, gerado por `c01_modelos_llm.ipynb`, com embeddings semânticos,
testar consultas de busca vetorial e analisar quando a recuperação funciona bem, quando
falha e como isso afeta a qualidade da resposta final gerada por um LLM.

<hr style="border:none;border-top:1px solid #cddffb;margin:14px 0;">

**🎯 Objetivos desta etapa**

- **Dividir** o corpus das aulas em chunks com sobreposição e metadados de rastreabilidade;
- **Indexar** os chunks com embeddings semânticos (`gemini-embedding-001`) num índice FAISS;
- **Testar** consultas de busca vetorial, incluindo a expansão de consulta com HyDE;
- **Analisar** quando a recuperação funciona bem, quando falha e por quê — usando o score como sinal;
- **Ligar** recuperação e geração, produzindo a resposta final a partir do contexto recuperado.

<hr style="border:none;border-top:1px solid #cddffb;margin:14px 0;">

**🔢 Decisão de modelo de embeddings**

Este notebook utiliza o modelo `gemini-embedding-001`, da mesma família da API Gemini já
empregada para geração em `c01_modelos_llm.ipynb` e `c02_prompting.ipynb`, via
`google-genai`. Essa escolha não introduz uma dependência nova ao projeto: a biblioteca
`google-genai` e a chave `GEMINI_API_KEY` já estavam configuradas desde c01 e c02, e o
modelo é executado inteiramente via API, sem exigir o download de um modelo local
adicional.

*Nota*: fixamos `output_dimensionality=768` para `gemini-embedding-001`, em vez do padrão
de 3072 dimensões desse modelo, o que reduz o custo de armazenamento e de cômputo.

<hr style="border:none;border-top:1px solid #cddffb;margin:14px 0;">

**💰 Justificativa econômica: por que utilizar a API da Gemini em vez de hospedar um modelo próprio**

Este notebook reproduz um padrão que um produto SaaS real utilizaria para responder
perguntas com base em documentos próprios: embeddings, busca vetorial e geração da resposta
final. Por esse motivo, cabe justificar a escolha de infraestrutura sob a ótica de um
produto em operação, e não apenas como exercício acadêmico.

A API paga por uso tem a vantagem de cobrar apenas pelos tokens efetivamente processados.
Atualmente, o Gemini Pro custa entre US$1,25 e US$2 por milhão de tokens de entrada e entre
US$10 e US$12 por milhão de tokens de saída, para contextos de até 200 mil tokens. Os
embeddings da Gemini têm custo bastante baixo, e o cache de contexto reduz em 80 a 90% o
custo de prompts repetidos, como o mesmo conjunto de chunks recuperados reutilizado em
várias chamadas.

A alternativa é hospedar um modelo próprio em um servidor com GPU, como uma instância
`g4dn.xlarge` na AWS, a um custo de aproximadamente US$0,526 por hora. Essa instância
permite executar um modelo open-source de cerca de 32 bilhões de parâmetros, comprimido e
quantizado, na GPU T4 disponível, a uma velocidade de 20 a 40 tokens por segundo. Essa GPU, e o custo mensal é fixo, em torno
de US$380 a US$400, independentemente do volume de uso.

No início de um SaaS, quando o número de clientes ainda é baixo, pagar por token é bem mais
barato e menos arriscado do que manter uma máquina ligada continuamente à espera de uso. A
conta só passa a favorecer o servidor próprio quando o volume se torna realmente alto, na
casa de dezenas de milhões de tokens por mês, ou quando existe uma exigência de privacidade
que impede o envio de dados a terceiros. Uma regra prática razoável é começar pela API e
considerar a migração para infraestrutura própria somente quando o gasto mensal com a
Gemini ultrapassar entre US$800 e US$1.500.

Por esse motivo, este notebook utiliza `gemini-3.5-flash`, e não `gemini-3.1-pro`, para
gerar texto. As duas tarefas de geração deste notebook — reescrever a consulta no formato
do transcript, na §3.1, e responder em até três frases a partir do contexto recuperado, na
§5 — são tarefas curtas e simples, exatamente o tipo de tarefa em que compensa utilizar o
modelo mais barato. Essa escolha difere da adotada em `c01_modelos_llm.ipynb`, onde o
Gemini Pro foi mantido mesmo custando cerca de 28 vezes mais por token que o Flash-Lite,
pois ali foi possível medir uma diferença real de qualidade na tradução, conforme as §6 a §8 de c01. Neste notebook não foi realizada uma comparação de qualidade equivalente para a
geração; sem essa evidência, não se justifica pagar o preço mais alto.

<hr style="border:none;border-top:1px solid #cddffb;margin:14px 0;">

**🌐 Convenções deste notebook**

- **Modelo de embeddings**: `gemini-embedding-001`, via `google-genai`, com
  `output_dimensionality=768` e `task_type` assimétrico: `RETRIEVAL_DOCUMENT` para indexar
  o corpus e `RETRIEVAL_QUERY` para as consultas. Essa é a prática correta para busca
  semântica, já que embeder uma pergunta curta não é a mesma tarefa que embeder um trecho
  de documento.
- **Geração**: `gemini-3.5-flash`, com os mesmos parâmetros de determinismo de c01 e c02,
  porém um modelo mais barato que o Pro utilizado em c01 — e o mesmo `gemini-3.5-flash`
  que `c02_prompting.ipynb` já adota para QA e sumarização — conforme a
  justificativa econômica apresentada acima. É reaproveitado tanto para expandir consultas
  na §3.1 quanto para gerar a resposta final, com Chain-of-Thought, a partir do contexto
  recuperado na §5.
- **Variáveis, funções e descrições em português**; consultas de teste e prompts em
  inglês, seguindo o mesmo critério de c02: modelos instruct seguem melhor instruções em
  inglês.

**Pré-requisito**: é necessário executar `c01_modelos_llm.ipynb` antes deste notebook, para
gerar os arquivos em `data/processed/*_portugues.txt`, usados como corpus.

</div>

In [1]:
%pip install google-genai faiss-cpu numpy pandas python-dotenv -q


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

MODELO_EMBEDDING = "gemini-embedding-001"
DIMENSAO_EMBEDDING = 768  # gemini-embedding-001 tem 3072 por padrão — fixamos 768 para
                          # reduzir custo de armazenamento/cômputo.
MODELO_GEMINI = "gemini-3.5-flash"


def gerar(system: str, user: str, max_new_tokens: int = 512) -> str:
    """Chama o Gemini 3.5 Flash para geração de texto — mesmo cliente e parâmetros de
    determinismo de c01/c02 (temperature=0, seed=42, thinking desligado — thinking_budget=0,
    como em c02: o thinking oculto consumia parte imprevisível do max_output_tokens e
    cortava as saídas curtas no meio); modelo mais
    barato que o Pro usado em c01, ver justificativa econômica na introdução deste notebook."""
    resposta = client.models.generate_content(
        model=MODELO_GEMINI,
        contents=user,
        config=types.GenerateContentConfig(
            system_instruction=system,
            temperature=0,
            seed=42,
            candidate_count=1,
            max_output_tokens=max_new_tokens,
            thinking_config=types.ThinkingConfig(thinking_budget=0),
        ),
    )
    return (resposta.text or "").strip()


def embed_lote(textos: list[str], task_type: str):
    """Embeda uma lista de textos (ex.: um lote de chunks) numa única chamada."""
    resposta = client.models.embed_content(
        model=MODELO_EMBEDDING, contents=textos,
        config=types.EmbedContentConfig(task_type=task_type, output_dimensionality=DIMENSAO_EMBEDDING),
    )
    return np.array([e.values for e in resposta.embeddings], dtype="float32")


def embed(texto: str, task_type: str):
    """Embeda um único texto (ex.: uma consulta) — reaproveita embed_lote() para não
    depender de um atributo de resposta diferente (.embedding, singular) que não existe
    nesta versão do SDK; só .embeddings (plural) é confirmado funcionando."""
    return embed_lote([texto], task_type)[0]


# Saídas estilizadas — mesma linguagem visual das caixas markdown do projeto (portado de
# c02, sem o parâmetro de idioma, que não existe neste notebook).
import re
import html as _html
from IPython.display import display, HTML


def exibir_titulo(texto):
    """Cabeçalho estilizado (mesma linguagem visual das caixas markdown do notebook)."""
    display(HTML(
        '<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:8px 14px;'
        'border-radius:4px;color:#111827;margin:10px 0 4px 0;font-weight:600;">'
        f'{_html.escape(texto)}</div>'
    ))


def exibir_texto(titulo, texto):
    """Cabeçalho (exibir_titulo) + texto gerado pelo modelo em caixa verde legível.
    Marcadores markdown do próprio modelo (**negrito**, headers "#") viram <strong> real."""
    exibir_titulo(titulo)
    texto_html = _html.escape(texto)
    texto_html = re.sub(r"^#{1,6}\s+(.+)$", r"<strong>\1</strong>", texto_html, flags=re.MULTILINE)
    texto_html = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", texto_html)
    texto_html = texto_html.replace("\n", "<br>")
    display(HTML(
        '<div style="background:#eafaf0;border-left:4px solid #2f9e5c;border-radius:4px;'
        'padding:8px 14px;margin:0 0 12px 0;'
        'font-family:system-ui,-apple-system,sans-serif;line-height:1.5;'
        'white-space:pre-wrap;color:#000000;">'
        f'{texto_html}</div>'
    ))


def exibir_aviso(texto, tipo="aviso"):
    """Mensagem de status estilizada no lugar de um print() solto. `tipo`: "aviso" (⚠️,
    âmbar), "erro" (❌, vermelho) ou "sucesso" (azul, mesma cor das outras caixas)."""
    cores = {
        "aviso": ("#fff8e6", "#d9a404"),
        "erro": ("#fdecea", "#c0392b"),
        "sucesso": ("#eef6ff", "#2b7de9"),
    }
    fundo, borda = cores.get(tipo, cores["aviso"])
    display(HTML(
        f'<div style="background:{fundo};border-left:4px solid {borda};padding:6px 14px;'
        'border-radius:4px;color:#111827;margin:4px 0;font-size:0.95em;">'
        f'{texto}</div>'
    ))

---

## §1 Corpus e chunking — sobreposição e metadados de rastreabilidade

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

O corpus é composto pelas 8 aulas já processadas por `c01_modelos_llm.ipynb`, disponíveis
em `data/processed/*_portugues.txt`, o texto original limpo, sem tradução. Cada linha do
transcript corresponde a uma fala isolada e curta, como `"Boa noite."` ou `"Estamos
desesperados."`; embeder linha a linha perderia contexto. Por esse motivo, o texto é
agrupado em janelas de `TAMANHO_CHUNK` linhas consecutivas, com `SOBREPOSICAO` linhas em
comum entre uma janela e a seguinte, em vez de blocos que se tocam apenas nas bordas.

A justificativa completa para essa escolha está no comentário do código a seguir.

</div>

In [3]:
from pathlib import Path

CORPUS_DIR = Path("data/processed")
TAMANHO_CHUNK = 5
SOBREPOSICAO = 2
STRIDE = TAMANHO_CHUNK - SOBREPOSICAO  # = 3


def montar_chunks(arquivo, sufixo_nome):
    """Divide um arquivo em janelas de TAMANHO_CHUNK linhas com SOBREPOSICAO entre elas,
    guardando metadados de rastreabilidade em cada chunk (não só o texto)."""
    linhas = arquivo.read_text(encoding="utf-8").splitlines()
    aula = arquivo.stem.replace(sufixo_nome, "")
    chunks = []
    for i in range(0, len(linhas), STRIDE):
        trecho = linhas[i:i + TAMANHO_CHUNK]
        chunks.append({
            "arquivo_origem": arquivo.name,
            "aula": aula,
            "chunk_id": f"{aula}#{i}",
            "linha_inicio": i,
            "linha_fim": i + len(trecho) - 1,
            "texto": "\n".join(trecho),
        })
    return chunks


# Modelos de linguagem e busca vetorial funcionam melhor com trechos menores — mas cortar
# sem sobreposição arrisca partir ao meio uma pergunta e sua resposta, ou uma fala e sua
# continuação, exatamente o tipo de falha de recuperação que testamos na §4. A sobreposição
# garante que uma troca curta apareça inteira em pelo menos um chunk, mesmo perto de uma
# borda. Cada chunk guarda de onde veio (aula, linha_inicio, linha_fim) para podermos
# apontar a origem exata de qualquer resultado recuperado (§4) ou citado na resposta final
# (§5).
arquivos_pt = sorted(CORPUS_DIR.glob("*_portugues.txt"))
exibir_aviso(f"📚 {len(arquivos_pt)} aulas encontradas em {CORPUS_DIR}", tipo="sucesso")

CHUNKS = []
for arquivo in arquivos_pt:
    CHUNKS.extend(montar_chunks(arquivo, "_portugues"))

exibir_aviso(
    f"🧩 {len(CHUNKS)} chunks no total (janelas de {TAMANHO_CHUNK} linhas, "
    f"sobreposição de {SOBREPOSICAO} linhas, passo de {STRIDE})",
    tipo="sucesso",
)


---

## §2 Indexação

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**Escolhemos o FAISS como índice vetorial deste notebook.** A busca semântica acontece
inteira dentro de um `faiss.IndexFlatIP` — biblioteca local, gratuita e em memória —, e
essa escolha é deliberada, não um padrão herdado: os motivos (custo zero, dependência já
existente, escala do corpus e simplicidade para quem avalia o projeto) estão detalhados
nos blocos abaixo, incluindo por que **não** usamos um banco de vetores hospedado.

Cada chunk é embedado com `task_type="RETRIEVAL_DOCUMENT"`, de forma assimétrica em relação
à consulta: um chunk de documento não é embedado da mesma forma que uma pergunta curta,
como discutido na §3. Os vetores são normalizados por L2 e armazenados em um
`faiss.IndexFlatIP`. O produto interno sobre vetores normalizados equivale à similaridade
de cosseno, o mesmo padrão já descrito em `CLAUDE.md` para o pipeline RAG planejado em c05.

**Por que FAISS**: é uma biblioteca local e gratuita, ao contrário da API de embeddings, que
cobra por token. `faiss-cpu` já é uma dependência pineada em `requirements.txt`, então não
introduz custo nem dependência nova ao projeto — o mesmo raciocínio já usado para justificar
`google-genai` na introdução deste notebook.

**Benefícios**: roda inteiramente em memória, sem precisar de um serviço de banco de vetores
hospedado, como Pinecone ou Weaviate, o que evita infraestrutura extra para um projeto deste
porte. Integra diretamente com arrays numpy, o mesmo formato já devolvido por `embed_lote()`.
E é o mesmo padrão de índice, produto interno sobre vetores normalizados, já planejado para o
pipeline RAG de `c05`, conforme `CLAUDE.md` — reaproveitar o padrão evita reinventar a lógica
de busca em cada notebook.

**Adequação a este caso de uso**: o corpus tem cerca de 2000 chunks (§1), uma escala pequena o
suficiente para que a busca exata por força bruta do `IndexFlatIP` já retorne em
milissegundos. Não há necessidade de um índice aproximado, como IVF, HNSW ou PQ, que trocaria
precisão por velocidade e exigiria ajuste de parâmetros extras sem benefício real neste
volume. Essas estruturas só compensam em corpora de milhões de vetores, o que não é o caso
aqui.

**E por que não Pinecone ou Qdrant?** Essas são opções mais robustas para busca vetorial, mas
o Pinecone é pago, e o Qdrant, mesmo sendo gratuito para hospedar, exige rodar um serviço à
parte, por exemplo em um contêiner Docker. Para este projeto, preferimos ficar com o FAISS
justamente porque ele roda em memória, dentro do próprio notebook, sem precisar instalar ou
configurar nenhum serviço extra — isso facilita para quem for avaliar o projeto, professor,
que só precisa instalar as dependências do `requirements.txt` e rodar o notebook, sem subir
nenhum contêiner à parte. Já em um cenário de produção real, a escolha seria outra: optaríamos
pelo Qdrant hospedado por nós mesmos, por ser mais robusto para operar de forma contínua, com
persistência em disco, atualização incremental dos vetores e por rodar como um serviço
independente da aplicação, algo que o FAISS, sendo só uma biblioteca em memória, não oferece.

**Persistência para c05**: ao final da indexação, o índice é salvo em disco
(`faiss.write_index`), junto com os metadados dos chunks e a configuração do modelo
de embeddings (`chunks_c03.json`). O pipeline RAG de c05 carrega esses artefatos
prontos em vez de reindexar — os embeddings do corpus são pagos uma única vez.

</div>

In [4]:
import json

import numpy as np
import faiss


def construir_indice(chunks, tamanho_lote=64):
    """Embeda os chunks em lotes (para reduzir o número de chamadas à API) e monta um
    índice FAISS de produto interno sobre vetores normalizados (= similaridade de
    cosseno)."""
    vetores = []
    for i in range(0, len(chunks), tamanho_lote):
        lote = chunks[i:i + tamanho_lote]
        vetores.append(embed_lote([c["texto"] for c in lote], "RETRIEVAL_DOCUMENT"))
        print(f"  embedando chunks {i}–{i + len(lote) - 1} de {len(chunks)}...")
    matriz = np.vstack(vetores)
    faiss.normalize_L2(matriz)
    indice = faiss.IndexFlatIP(matriz.shape[1])
    indice.add(matriz)
    return indice


exibir_aviso("⏳ Indexando corpus em português (pode levar alguns minutos)...")
INDICE = construir_indice(CHUNKS)
exibir_aviso(f"✅ Índice pronto: {INDICE.ntotal} vetores de dimensão {INDICE.d}", tipo="sucesso")

# Persistência para c05: o FAISS serializa o índice inteiro com write_index/read_index,
# mas não guarda metadados — por isso os chunks e a configuração do modelo de embeddings
# vão num JSON ao lado. c05 carrega os dois arquivos prontos em vez de reindexar.
ARQUIVO_INDICE = CORPUS_DIR / "indice_faiss_c03.index"
ARQUIVO_CHUNKS = CORPUS_DIR / "chunks_c03.json"
faiss.write_index(INDICE, str(ARQUIVO_INDICE))
ARQUIVO_CHUNKS.write_text(json.dumps({
    "modelo_embedding": MODELO_EMBEDDING,
    "dimensao_embedding": DIMENSAO_EMBEDDING,
    "task_type_documento": "RETRIEVAL_DOCUMENT",
    "task_type_consulta": "RETRIEVAL_QUERY",
    "tamanho_chunk": TAMANHO_CHUNK,
    "sobreposicao": SOBREPOSICAO,
    "chunks": CHUNKS,
}, ensure_ascii=False), encoding="utf-8")
exibir_aviso(
    f"💾 Artefatos salvos para reuso em c05: {ARQUIVO_INDICE.name} e {ARQUIVO_CHUNKS.name}",
    tipo="sucesso",
)


  embedando chunks 0–63 de 2173...
  embedando chunks 64–127 de 2173...
  embedando chunks 128–191 de 2173...
  embedando chunks 192–255 de 2173...
  embedando chunks 256–319 de 2173...
  embedando chunks 320–383 de 2173...
  embedando chunks 384–447 de 2173...
  embedando chunks 448–511 de 2173...
  embedando chunks 512–575 de 2173...
  embedando chunks 576–639 de 2173...
  embedando chunks 640–703 de 2173...
  embedando chunks 704–767 de 2173...
  embedando chunks 768–831 de 2173...
  embedando chunks 832–895 de 2173...
  embedando chunks 896–959 de 2173...
  embedando chunks 960–1023 de 2173...
  embedando chunks 1024–1087 de 2173...
  embedando chunks 1088–1151 de 2173...
  embedando chunks 1152–1215 de 2173...
  embedando chunks 1216–1279 de 2173...
  embedando chunks 1280–1343 de 2173...
  embedando chunks 1344–1407 de 2173...
  embedando chunks 1408–1471 de 2173...
  embedando chunks 1472–1535 de 2173...
  embedando chunks 1536–1599 de 2173...
  embedando chunks 1600–1663 de 217

---

## §3 Consultas de teste

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

O conjunto de consultas de teste foi montado deliberadamente variado, para permitir
discutir tanto o sucesso quanto a falha da recuperação com casos concretos na §4:

1. **Deveria funcionar bem**: consulta sobre um tópico específico e bem coberto pelo
   corpus, as ferramentas do curso.
2. **Testa robustez a erros de transcrição**: utiliza o termo correto, "Hugging Face" e
   "CUDA", para verificar se o embedding ainda recupera a linha em que a transcrição saiu
   errada, como "Ruginface" e "cuida". Está diretamente ligada aos achados de fidelidade já
   documentados na §6 de c01.
3. **Fora do corpus, consulta adversarial**: aborda um assunto que a disciplina nunca
   menciona, para verificar se o score de similaridade cai visivelmente, sinal de que não
   há uma resposta boa disponível.
4. **Logística recorrente entre aulas**: verifica se o top-k recuperado cruza aulas
   diferentes.

</div>

In [5]:
def buscar(texto_consulta, indice, chunks_meta, k=3):
    """Embeda a consulta com task_type=RETRIEVAL_QUERY (assimétrico em relação ao
    documento) e devolve os k chunks mais similares, com score de similaridade."""
    vetor = embed(texto_consulta, "RETRIEVAL_QUERY").reshape(1, -1)
    faiss.normalize_L2(vetor)
    scores, posicoes = indice.search(vetor, k)
    return [{**chunks_meta[pos], "score": float(score)} for score, pos in zip(scores[0], posicoes[0])]


# "consulta" (em inglês) é a entrada do pipeline — embedding e prompts, convenção do
# projeto; "consulta_es"/"consulta_pt" são só para exibição nos banners e justificativas.
CONSULTAS_TESTE = [
    {
        "id": 1,
        "consulta": "What tools and libraries does the course use to access language models?",
        "consulta_es": "¿Qué herramientas y bibliotecas usa el curso para acceder a los modelos de lenguaje?",
        "consulta_pt": "Quais ferramentas e bibliotecas o curso usa para acessar os modelos de linguagem?",
        "categoria": "deveria funcionar bem",
    },
    {
        "id": 2,
        "consulta": "Hugging Face and CUDA",
        "consulta_es": "Hugging Face y CUDA",
        "consulta_pt": "Hugging Face e CUDA",
        "categoria": "robustez a erros de transcrição",
    },
    {
        "id": 3,
        "consulta": "What is the recommended treatment for chronic kidney disease in cats?",
        "consulta_es": "¿Cuál es el tratamiento recomendado para la enfermedad renal crónica en gatos?",
        "consulta_pt": "Qual é o tratamento recomendado para a doença renal crônica em gatos?",
        "categoria": "fora do corpus (adversarial)",
    },
    {
        "id": 4,
        "consulta": "How does the professor say students should submit their assignment?",
        "consulta_es": "¿Cómo dice el profesor que los alumnos deben entregar el trabajo?",
        "consulta_pt": "Como o professor diz que os alunos devem entregar o trabalho?",
        "categoria": "logística recorrente entre aulas",
    },
]


### §3.1 Prompt engineering aplicado às consultas — expansão de consulta com HyDE

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

Antes de embeder a consulta crua, pedimos ao Gemini para reescrevê-la como um trecho
hipotético de fala de aula que a responderia, técnica conhecida como *Hypothetical
Document Embeddings*, ou HyDE.

A justificativa completa para essa escolha está no comentário do código a seguir.

</div>

In [6]:
SYSTEM_HYDE = "You are simulating what a spoken lecture transcript sounds like."


def expandir_consulta_hyde(consulta):
    prompt = (
        "Rewrite the following question as a short hypothetical excerpt of spoken dialogue "
        "that might appear in a class transcript answering it, using natural spoken "
        "language (not a formal written answer). Return ONLY the hypothetical excerpt, in "
        f"English, with no extra text.\n\nQuestion: {consulta}"
    )
    return gerar(SYSTEM_HYDE, prompt, max_new_tokens=512)


# Isto é prompt engineering aplicado à RECUPERAÇÃO, não só à resposta final (§5) — uma
# pergunta curta e formal ("What tools...") tem uma forma muito diferente do que aparece de
# fato no transcript (falas soltas, no meio de uma conversa). Um trecho no mesmo "formato"
# do que estamos buscando tende a ficar mais perto, no espaço de embeddings, do chunk certo
# do que a pergunta original. Comparamos consulta crua vs. expandida só para as consultas
# #1 e #2 (as duas onde já sabemos, por outros meios, o que seria uma boa recuperação) — e
# não assumimos que a expansão sempre ajuda: se para alguma delas não mudar nada ou piorar,
# isso também entra na análise da §4 como um resultado honesto, não escondido.
def executar_consulta(item, indice, chunks_meta, k=3):
    resultado = {
        "id": item["id"], "consulta": item["consulta"], "categoria": item["categoria"],
        "consulta_es": item["consulta_es"], "consulta_pt": item["consulta_pt"],
        "bruta": buscar(item["consulta"], indice, chunks_meta, k=k),
    }
    if item["id"] in (1, 2):
        consulta_expandida = expandir_consulta_hyde(item["consulta"])
        resultado["consulta_expandida"] = consulta_expandida
        resultado["expandida"] = buscar(consulta_expandida, indice, chunks_meta, k=k)
    return resultado


RESULTADOS = [executar_consulta(item, INDICE, CHUNKS, k=3) for item in CONSULTAS_TESTE]


---

## §4 Análise dos resultados

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

Para cada consulta é apresentada uma tabela com os top-k chunks recuperados: aula, linhas,
texto e score de similaridade, usando os metadados de rastreabilidade descritos na §1. Para
as consultas 1 e 2, o resultado da consulta crua é comparado lado a lado com o da consulta
expandida, apresentada na §3.1.

Cada linha da tabela traz também uma coluna **Resumo gerado por Gemini**: o texto do
chunk recuperado é usado como input de um prompt ao `gemini-3.5-flash` (o mesmo modelo
de geração deste notebook), que resume em poucas frases (2 a 3) o que aquele trecho
diz em relação à pergunta.
Assim a leitura em prosa e a evidência — texto, scores e rastreabilidade — ficam na
mesma tabela.

</div>

In [7]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", None)


PREFIXO_AULA = "Sistemas_Cognitivos_com_LLMs_aula-"


def mostrar_tabela(df, largura_padrao=480, larguras=None):
    """Mostra um DataFrame com o texto das células por extenso, quebrando linha em vez de
    truncar — mesma linguagem visual (azul #eef6ff) do restante do projeto, no padrão da
    exibir_tabela de c02: fundo nas células, largura máxima por coluna, índice começando
    em 1 e score com 3 casas decimais. O prefixo comum do nome da aula é removido só na
    exibição, para a coluna caber sem quebrar em cada underscore."""
    df = df.copy()
    if "aula" in df.columns:
        df["aula"] = df["aula"].str.replace(PREFIXO_AULA, "", regex=False)
    if list(df.index) == list(range(len(df))):
        df.index = range(1, len(df) + 1)
    larguras = larguras or {"texto": 420, "Resumo gerado por Gemini": 420, "aula": 150,
                            "linha_inicio": 60, "linha_fim": 60, "score": 70}
    estilos_coluna = [
        {"selector": f".col{i}", "props": [("max-width", f"{larguras.get(col, largura_padrao)}px")]}
        for i, col in enumerate(df.columns)
    ]
    estilo = df.style.set_properties(**{
        "text-align": "left",
        "white-space": "pre-wrap",
        "vertical-align": "top",
        "background-color": "#f7fbff",
        "color": "#111827",
    }).set_table_styles([
        {"selector": "th", "props": [("text-align", "left"), ("background-color", "#eef6ff")]},
        *estilos_coluna,
    ])
    if "score" in df.columns:
        estilo = estilo.format({"score": "{:.3f}"})
    display(estilo)


def resumir_chunk(consulta, chunk):
    """Usa o texto de UM chunk recuperado como input de um prompt ao gemini-3.5-flash e
    devolve um resumo de 2 a 3 frases, em português, do que o trecho diz em relação à
    pergunta — vira a coluna "Resumo gerado por Gemini" da tabela de resultados."""
    system = (
        "You summarize one excerpt retrieved from a class transcript."
        f"\n<excerpt>\n{chunk['texto']}\n</excerpt>"
    )
    user = (
        "Summarize in Portuguese, in 2 to 3 sentences, what this excerpt says in "
        "relation to the question below. If the excerpt does not relate to the question, "
        f"say so explicitly. Return ONLY the summary.\n\nQuestion: {consulta}"
    )
    return gerar(system, user, max_new_tokens=512)


COLUNAS = ["aula", "linha_inicio", "linha_fim", "texto", "score"]


def tabela_com_resumo(consulta, chunks):
    """Tabela de resultados com a coluna "Resumo gerado por Gemini" (2 a 3 frases por
    chunk, geradas pelo modelo) entre o texto recuperado e o score."""
    df = pd.DataFrame(chunks)[COLUNAS]
    df["Resumo gerado por Gemini"] = [resumir_chunk(consulta, c) for c in chunks]
    return df[["aula", "linha_inicio", "linha_fim", "texto", "Resumo gerado por Gemini", "score"]]

def exibir_consulta(numero, total, r):
    """Banner numerado de consulta — mesmo estilo visual dos banners de pergunta de c05,
    com margem superior grande para separar um bloco de consulta do anterior."""
    display(HTML(
        '<div style="background:#2b7de9;color:#ffffff;border-radius:6px;'
        'padding:10px 14px;margin:28px 0 6px 0;'
        'font-family:system-ui,-apple-system,sans-serif;line-height:1.5;">'
        f'<span style="background:#ffffff;color:#2b7de9;border-radius:12px;'
        f'padding:2px 10px;font-weight:700;font-size:0.85em;margin-right:10px;'
        f'white-space:nowrap;">Consulta {numero} de {total}</span>'
        f'<strong>{_html.escape(r["consulta_es"])}</strong> '
        f'<span style="opacity:0.85;">({_html.escape(r["consulta_pt"])})</span><br>'
        f'<span style="opacity:0.85;font-size:0.95em;">categoria: {_html.escape(r["categoria"])}</span>'
        '</div>'
    ))


for r in RESULTADOS:
    exibir_consulta(r["id"], len(RESULTADOS), r)
    exibir_aviso("Resultado (consulta crua):", tipo="sucesso")
    mostrar_tabela(tabela_com_resumo(r["consulta"], r["bruta"]))
    if "expandida" in r:
        exibir_texto("Consulta expandida (HyDE)", r["consulta_expandida"])
        exibir_aviso("Resultado (consulta expandida):", tipo="sucesso")
        mostrar_tabela(tabela_com_resumo(r["consulta"], r["expandida"]))


,aula,linha_inicio,linha_fim,texto,Resumo gerado por Gemini,score
1,23-06-2026_20-08,228,232,"Fernando Guimarães Ferreira: Está na apresentação. Isso vou botar todas as bibliotecas do Langchen que a gente vai precisar. Fernando Guimarães Ferreira: A gente vai precisar das interfaces com hangface para fazer o embeding. A gente vai precisar da interface com Chroma para poder ler o vetor. A gente vai precisar da interface com o Lhama Fernando Guimarães Ferreira: para fazer. A gente vai precisar dos documentos. E aí algumas questões aqui com template vector Store. Então essas são as bibliotecas, como eu falei para vocês que são necessárias. Fernando Guimarães Ferreira: Outro ponto que a gente vai precisar Fernando Guimarães Ferreira: é a instalação do Oleama. O que é Oleama? Oleama é um","O trecho indica que o curso utiliza a biblioteca LangChain para integrar as ferramentas necessárias para acessar e manipular os modelos de linguagem. Especificamente, são mencionadas as interfaces com o Hugging Face para geração de embeddings, com o Chroma para leitura de vetores e com o Ollama para a execução dos modelos. Além disso, o apresentador destaca a necessidade de instalar o Ollama e utilizar templates de documentos e armazenamento de vetores.",0.720
2,03-06-2026_20-10,744,748,"Fernando Guimarães Ferreira: Basicamente isso. E aí, qual tecnologia vocês vão usar Fernando Guimarães Ferreira: é uma opção que vocês vão falar. Você pode utilizar o Open air ruginface, modelos, pachados, Gpt Forol. Qualquer alternativa, Fernando Guimarães Ferreira: alguma dessas alternativas. Fernando Guimarães Ferreira: Baseado no que vocês vão aprender nas próximas aulas. Nessa aula inclusa Fernando Guimarães Ferreira: então tarefa você deverá desenvolver uma aplicação funcional com Lms em Python. A solução deve representar um sistema aplicado e não apenas uma sequência de testes isolados.","O curso permite a utilização de diversas tecnologias e bibliotecas para acessar os modelos de linguagem, incluindo OpenAI, Hugging Face, modelos baixados localmente e GPT4All. A escolha da ferramenta cabe aos alunos, com base no conteúdo ensinado ao longo das aulas. O objetivo final é que os estudantes utilizem essas alternativas para desenvolver uma aplicação funcional em Python.",0.714
3,01-06-2026_20-08,111,115,"Fernando Guimarães Ferreira: enquanto a gente tiver o convênio, Fernando Guimarães Ferreira: então a gente vai como ferramental trabalhar com ruginface, com Api dessas Fernando Guimarães Ferreira: de modelos da Openai ou Gemini. Enfim, a gente trabalhar com essas possibilidades. Fernando Guimarães Ferreira: Eu vou trabalhar com uma ferramenta instalada localmente, mais para o final do curso chamada Gpt Four All. Fernando Guimarães Ferreira: E depois a gente vai estruturar com,","Para acessar os modelos de linguagem, o curso utilizará como ferramentas o Hugging Face e APIs de provedores como OpenAI ou Gemini, aproveitando o convênio existente. Além disso, mais para o final do curso, será trabalhada uma ferramenta instalada localmente chamada GPT4All.",0.705


,aula,linha_inicio,linha_fim,texto,Resumo gerado por Gemini,score
1,01-06-2026_20-08,117,121,"Fernando Guimarães Ferreira: para vocês poderem consumir. Então essa é a nossa perspectiva. Alguém pode levantar a mão e falar assim. Ah, Langchain ou cru ai, ou sei lá, ferramentas voltadas para agentes. Não é o foco desse bloco. A gente vai ter um bloco Fernando Guimarães Ferreira: construído para desenvolvimento de agentes. Não é o foco desse bloco, lembrando que o próximo bloco que vocês têm é um bloco de redes neurais, Fernando Guimarães Ferreira: Deep learning, né? Fernando Guimarães Ferreira: Então o mapeamento é exatamente esse. Aqui. A gente vai falar dos modelos pré treinados. Próximo bloco, vocês entendem como esses modelos são construídos. Basicamente, a gente vai falar aqui. Um pouco de arquitetura e tudo mais, mas não é o foco Fernando Guimarães Ferreira: não é o foco de desenvolver um modelo de linguagem.","O trecho não menciona quais ferramentas e bibliotecas específicas são utilizadas no curso para acessar os modelos de linguagem. O palestrante apenas esclarece que o foco do bloco atual são os modelos pré-treinados e suas arquiteturas, e não o desenvolvimento de agentes com ferramentas como LangChain ou CrewAI, que serão abordadas em outro momento.",0.714
2,01-06-2026_20-08,114,118,"Fernando Guimarães Ferreira: Eu vou trabalhar com uma ferramenta instalada localmente, mais para o final do curso chamada Gpt Four All. Fernando Guimarães Ferreira: E depois a gente vai estruturar com, Fernando Guimarães Ferreira: enfim, com ferramentas open source, um Rag Fernando Guimarães Ferreira: para vocês poderem consumir. Então essa é a nossa perspectiva. Alguém pode levantar a mão e falar assim. Ah, Langchain ou cru ai, ou sei lá, ferramentas voltadas para agentes. Não é o foco desse bloco. A gente vai ter um bloco Fernando Guimarães Ferreira: construído para desenvolvimento de agentes. Não é o foco desse bloco, lembrando que o próximo bloco que vocês têm é um bloco de redes neurais,","O curso utilizará a ferramenta instalada localmente chamada GPT4All, além de ferramentas de código aberto (open source) para a estruturação de um sistema RAG. O professor esclarece que ferramentas voltadas para o desenvolvimento de agentes, como LangChain ou CrewAI, não serão o foco deste bloco específico do curso.",0.708
3,03-06-2026_20-10,738,742,"Fernando Guimarães Ferreira: Bom, eu acho que para ler tudo que a gente tem para fazer, a gente levaria mais tempo para explicar todos os pontos. Fernando Guimarães Ferreira: Mas o básico que vocês vão fazer. Como pode construir uma aplicação capaz de receber entrada de linguagem, acessar, processar Fernando Guimarães Ferreira: e produzir respostas úteis e estruturadas. Essa que é a ideia que a gente vai fazer, lembrando que a gente tem o processamento de modelos. Fernando Guimarães Ferreira: A gente tem, a partir da próxima aula. Duas aulas de Prompt engineering. Fernando Guimarães Ferreira: Depois a gente entra num bloco sobre hag, não é?","O trecho apresentado não menciona quais ferramentas e bibliotecas específicas o curso utiliza para acessar os modelos de linguagem. O texto apenas indica que os alunos aprenderão a construir uma aplicação capaz de processar linguagem e produzir respostas estruturadas, além de detalhar que o cronograma incluirá aulas de engenharia de prompt e RAG (geração aumentada por recuperação).",0.704


,aula,linha_inicio,linha_fim,texto,Resumo gerado por Gemini,score
1,03-06-2026_20-10,699,703,Fernando Guimarães Ferreira: Oi. kevin.rodrigues: Esses modelos aí. Roda no Gpu. Fernando Guimarães Ferreira: Em Gpu. É isso que você quer dizer? kevin.rodrigues: Isso. Fernando Guimarães Ferreira: É? É tudo Gpu. Funciona mais rápido em Gpu.,"O trecho confirma que os modelos mencionados rodam em GPU, destacando que o funcionamento é mais rápido nesse tipo de hardware. Embora não cite explicitamente os termos ""Hugging Face"" ou ""CUDA"", a discussão sobre a execução de modelos em GPU está diretamente relacionada ao uso de CUDA para aceleração de hardware nesse contexto.",0.685
2,22-06-2026_20-11,42,46,"pedro.duarte: Eu estou executando tudo pelo Collab ali, né? Eu não cheguei a fazer um teste testando os mesmos modelos ali com os prompt sem usar gpu, né? pedro.duarte: Aí talvez seria um caso, né? Ver também sem ser pela Gpu e comparar, né? Porque ali ele tem os tempos de execução. Tudo de cada modelo ali, né? Fernando Guimarães Ferreira: É outra. pedro.duarte: Com a. Fernando Guimarães Ferreira: Ou, às vezes, pela própria documentação do que está ali no ruginface, por exemplo, dos requisitos necessários. Não necessariamente. Estou pedindo para vocês executarem vários códigos. Mas assim, um estudo do tipo não, esse aqui é o suficiente para rodar, por exemplo, online, ou esse aqui daria para rodar online e local no computador. Esse aqui precisa. É muito grande. Precisa de uma","O estudante Pedro Duarte menciona que está executando os modelos do Hugging Face no Google Colab utilizando GPU e sugere realizar testes comparativos de tempo de execução sem o uso desse recurso. Em resposta, o professor Fernando Guimarães Ferreira complementa que, além dos testes práticos, os alunos podem consultar a própria documentação do Hugging Face para analisar os requisitos de hardware necessários para rodar os modelos localmente ou online.",0.679
3,03-06-2026_20-10,243,247,"Fernando Guimarães Ferreira: e ele tem também. Alguém estava falando aí de computador e tudo mais. Ele tem playground também para rodar coisas. Quando a gente entra aqui no Ruginface. Fernando Guimarães Ferreira: Ele tem Spaces. Fernando Guimarães Ferreira: Então você pode criar ou usar Fernando Guimarães Ferreira: computing daqui, né? E o Collab? Ele tem Gpu, né? Também é uma coisa que é importante, principalmente pra essa disciplina. Para a gente rodar coisas. Fernando Guimarães Ferreira: Eu vou mostrar para vocês como fazer, mas habilitar a Gpu do Collab facilita bastante. Não é uma Gpu irrestrita, não é uma Gpu que vai funcionar durante vinte e quatro horas. Você rodando alguma coisa. Mas para rodar, alguma coisa funciona","O trecho aborda o Hugging Face mencionando que a plataforma possui um *playground* e a funcionalidade de ""Spaces"" para rodar aplicações. Embora não mencione diretamente a tecnologia CUDA, o palestrante destaca a importância do uso de GPUs, citando como exemplo a possibilidade de habilitar esse recurso no Google Colab para facilitar a execução de tarefas da disciplina.",0.673


,aula,linha_inicio,linha_fim,texto,Resumo gerado por Gemini,score
1,03-06-2026_20-10,702,706,"kevin.rodrigues: Isso. Fernando Guimarães Ferreira: É? É tudo Gpu. Funciona mais rápido em Gpu. Fernando Guimarães Ferreira: Funciona bem mais rápido em Japão, Fernando Guimarães Ferreira: são modelos que se implementada em gpor são na melhores. Fernando Guimarães Ferreira: Bom, uma pergunta que vocês poderiam me fazer. É isso que está escrito aqui embaixo? Pô, professor tem um Pipeline. Mas também tem o modelo. Quando é que a gente usa um? Quando a gente usa o outro, né? Lá no início da aula a gente.","O trecho aborda o uso de GPUs para a execução de modelos, destacando que eles funcionam de forma significativamente mais rápida quando implementados nessa tecnologia. Embora não mencione explicitamente os termos ""Hugging Face"" ou ""CUDA"", a discussão contextualiza a eficiência de processamento de modelos em hardware acelerado. O professor também inicia uma explicação sobre a diferença de uso entre um ""Pipeline"" e um ""modelo"".",0.707
2,17-06-2026_20-05,90,94,"Fernando Guimarães Ferreira: Senão não é que os códigos não vão rodar, mas vão demorar muito para rodar. Fernando Guimarães Ferreira: Então, lembrando aqui os passos, né? Chega aqui. Fernando Guimarães Ferreira: É, não peraí antes de fazer aqui Checkand, né? Muda para T4 Gpu. Fernando Guimarães Ferreira: Conecta Fernando Guimarães Ferreira: aí. Talvez ele reclame aqui porque eu tava usando antes. Aí é só eu matar aqui o que eu tava usando antes",O trecho apresentado não faz menção direta ao Hugging Face ou ao CUDA. O palestrante apenas orienta sobre a necessidade de alterar o ambiente de execução para uma GPU T4 e conectar-se a ela para evitar que os códigos demorem a rodar.,0.698
3,03-06-2026_20-10,261,265,"Fernando Guimarães Ferreira: Então eu falei do Gpu. Fernando Guimarães Ferreira: Vocês vêm aqui nesse cantinho aqui vocês vão ver que existe Change Run Time. Fernando Guimarães Ferreira: E aqui vocês podem pegar uma Gpu, T4 e Gpu. Fernando Guimarães Ferreira: No momento que vocês fazem isso, Salmo: Fernando Guimarães Ferreira: e aí basta clicar em conectar. Está salvando. Basta clicar em conectar,","O trecho apresentado não faz menção direta ao Hugging Face ou ao CUDA. O palestrante apenas explica como alterar o tipo de ambiente de execução (Run Time) para utilizar uma GPU T4 e, em seguida, salvar e conectar a ela.",0.695


,aula,linha_inicio,linha_fim,texto,Resumo gerado por Gemini,score
1,25-06-2026_20-13,861,865,"Fernando Guimarães Ferreira: Não basta! Efrain Colmenares: Cicatricure. Fernando Guimarães Ferreira: Temos outros professores. Não vai ser um professor que está ansiosíssimo para dar aula para vocês, Fernando Guimarães Ferreira: mas eu acompanho então qualquer dúvida. Qualquer ponto é só me chamar. Fernando Guimarães Ferreira: Tá bom, é isso, pessoal.",O trecho apresentado não faz nenhuma referência ao tratamento de doença renal crônica em gatos. O diálogo aborda apenas a dinâmica de professores e o acompanhamento de dúvidas dos alunos.,0.530
2,08-06-2026_20-07,168,172,"Fernando Guimarães Ferreira: Pode falar. Kevin tinha levantado a mão. kevin.rodrigues: É que eu ia perguntar se reggae consome kevin.rodrigues: contexto da mesma forma que os promotes normais. kevin.rodrigues: Só que eu pensei que o Wreck tá vitorizado. Ele tá em forma de embeding já, né? Fernando Guimarães Ferreira: Ele está vitorizado aí na aula de rádio. A gente vai falar melhor sobre isso, mas o que vai acontecer é que","O trecho apresentado não possui nenhuma relação com a pergunta sobre o tratamento de doença renal crônica em gatos. O diálogo aborda uma dúvida técnica sobre o consumo de contexto por ""reggae"" (RAG) e vetorização/embeddings em inteligência artificial.",0.525
3,25-06-2026_20-13,858,862,Efrain Colmenares: Para a. Fernando Guimarães Ferreira: Não aguentam tanto tempo assim. Não. Efrain Colmenares: E aí? Fernando Guimarães Ferreira: Não basta! Efrain Colmenares: Cicatricure.,"O trecho apresentado não faz nenhuma referência ao tratamento de doença renal crônica em gatos. A breve conversa entre os interlocutores menciona apenas uma discussão sobre tempo de resistência e uma referência informal ao produto ""Cicatricure"".",0.525


,aula,linha_inicio,linha_fim,texto,Resumo gerado por Gemini,score
1,23-06-2026_20-08,651,655,"Fernando Guimarães Ferreira: A gente tem plano de botar todas as entregas em algum momento, com vídeos com os alunos apresentando, mas eu não implementamos isso ainda. Então, talvez daqui o tempo isso seja até uma prática. fabricio.gouveia: É um. Fernando Guimarães Ferreira: É bom. Vocês viram aqui, né? Como que eu fiz, né? Simplesmente. pedro.duarte: Sim, sim. Fernando Guimarães Ferreira: Vamos fazer um formulário. Isso aqui inclusive, pode","O trecho não explica como os alunos devem enviar o trabalho, mencionando apenas que há planos futuros de incluir todas as entregas com vídeos de apresentação dos estudantes. No momento da gravação, o professor apenas demonstra a criação de um formulário, sem detalhar o processo de envio atual.",0.665
2,23-06-2026_20-08,648,652,"Fernando Guimarães Ferreira: Mais fácil é o colapso. Fernando Guimarães Ferreira: Se por acaso acharem que o Collab não tem recurso, faz um visual Code e me manda o código. Fernando Guimarães Ferreira: Quem quiser gravar um vídeo, eu não me oponho. Não fiquem à vontade para apresentar para. Enfim, mas eu não botei como requisito. Fernando Guimarães Ferreira: A gente tem plano de botar todas as entregas em algum momento, com vídeos com os alunos apresentando, mas eu não implementamos isso ainda. Então, talvez daqui o tempo isso seja até uma prática. fabricio.gouveia: É um.","O professor afirma que os alunos podem enviar o código feito no Google Colab ou, caso prefiram, utilizar o Visual Studio Code e enviar o arquivo correspondente. Além disso, ele menciona que, embora não seja um requisito obrigatório para esta entrega, os alunos que desejarem podem gravar um vídeo apresentando o trabalho.",0.664
3,08-06-2026_20-07,762,766,"pedro.duarte: tem alguma forma de, tipo, eu disponibilizar essa documentação, mas eu não enviar o documento, professor ou algo assim. Algo tive que, tipo, só meio que por debaixo dos panos. A gente tem alguma forma de fazer isso? A gente vai ver algo assim ou. Fernando Guimarães Ferreira: Para disponibilizar para mim. pedro.duarte: É. Fernando Guimarães Ferreira: Casos específicos. Assim, vocês podem disponibilizar. Eu mostro, mas vocês podem disponibilizar o próprio binário do do Fernando Guimarães Ferreira: enfim, dos modelos aqui da.","O estudante pergunta se há uma forma de disponibilizar a documentação do trabalho de maneira mais reservada, sem enviar o documento diretamente. O professor responde que, em casos específicos, os alunos podem disponibilizar o próprio binário dos modelos para que ele possa visualizar.",0.656


<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**📋 Discussão dos resultados, com `gemini-3.5-flash` na geração**

A consulta 1, que deveria funcionar bem, teve seu trecho correto recuperado pela busca
crua: o top-1 obteve score de cerca de 0,72 e trouxe as bibliotecas do curso, LangChain, Hugging
Face, Chroma, Llama e Ollama. Ao aplicar HyDE, o Flash reescreveu a pergunta com foco
específico na biblioteca `transformers` do Hugging Face, o que elevou o score para cerca de 0,74,
superior ao da consulta crua. O chunk retornado, no entanto, passou a ser mais estreito,
tratando especificamente do acesso a modelos via Hugging Face, e não da lista completa de
ferramentas. A expansão, portanto, aumentou a confiança da busca ao custo de restringir o
foco do resultado.

Na consulta 2, sobre robustez a erros de transcrição, a busca crua já localiza o assunto
correto, GPU, com score de cerca de 0,69, e o segundo resultado já menciona "ruginface", com score
de cerca de 0,68. Com HyDE, o Flash produziu um trecho mais literal sobre CUDA e aceleração de GPU.
O resultado foi que o top-1 da busca expandida passou a ser o mesmo chunk já encontrado
pela busca crua, com score mais alto, cerca de 0,71, contra cerca de 0,69 antes, mas sem trazer
nenhum chunk com menção explícita a "Ruginface" ou "Huginface" entre os três primeiros.
A expansão foi gerada com `gemini-3.5-flash`, o modelo escolhido neste notebook por
custo e eficiência. Esse resultado evidencia que a eficácia do HyDE depende
também de como o modelo de geração redige o trecho hipotético: o texto produzido pelo Flash
permaneceu mais literal sobre CUDA e GPU, sem coincidir com o vocabulário
do erro de transcrição.

Na consulta 3, fora do corpus, o score caiu de forma visível, no máximo cerca de 0,53, bem abaixo
da faixa aproximada de 0,65 a 0,74 observada nas demais consultas. Como essa consulta não utiliza
geração, o resultado é idêntico ao de execuções anteriores. Esse resultado confirma a
hipótese de que o índice não atribui confiança falsa a um resultado quando a pergunta é
genuinamente estranha ao corpus.

Na consulta 4, sobre logística entre aulas, o top-3 trouxe chunks de duas aulas diferentes:
a aula de 23 de junho aparece duas vezes, e a aula de 8 de junho, uma vez. As duas
primeiras posições, contudo, vêm da mesma aula, em linhas próximas umas das outras, de modo que a diversidade entre aulas aparece apenas na terceira posição, e não já no
top-2. Esse resultado também é idêntico ao de execuções anteriores, uma vez que essa
consulta não utiliza geração.

As hipóteses gerais para explicar qualquer falha observada são: chunk curto demais mesmo
com sobreposição; vocabulário distorcido pela transcrição do áudio, um problema da fonte
que não é corrigível por uma busca melhor; consulta genuinamente fora do corpus; ou
informação relevante que, ainda assim, ficou dividida entre chunks.

</div>

---

## §5 Da recuperação à resposta final

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**O que esta seção faz.** Aqui a recuperação vira resposta: os chunks encontrados pela
busca são entregues ao Gemini como único contexto permitido, e o modelo responde a
pergunta a partir deles. Para mostrar os dois lados, comparamos o melhor e o pior caso da
§4: a consulta 1, cuja recuperação foi boa, e a consulta 3, fora do corpus, cuja
recuperação foi ruim de propósito.

**Como a resposta é gerada.** O prompt usa **Chain-of-Thought**, o mesmo padrão de c02
(§1.4): o modelo primeiro cita trechos curtos dos chunks que vai usar, depois numera os
passos do raciocínio, e só então dá a resposta final numa linha iniciada por `RESPOSTA:`,
que o código extrai por regex. O raciocínio visível é o que permite auditar a resposta:
dá para conferir, linha a linha, se ela saiu mesmo do contexto recuperado.

**E quando o formato falha.** Se o marcador `RESPOSTA:` não vier (por exemplo, saída
cortada por falta de tokens), o problema é reenviado ao modelo numa retentativa única,
pedindo uma versão mais curta — o mesmo tratamento de erro de `gerar_json` em c02.

</div>

In [8]:
import re

MARCADOR_RESPOSTA = "RESPOSTA:"


def montar_contexto(resultados_busca):
    return "\n\n".join(
        f'[{r["aula"]} linhas {r["linha_inicio"]}-{r["linha_fim"]}]\n{r["texto"]}'
        for r in resultados_busca
    )


def responder_com_contexto(consulta, resultados_busca, max_tentativas=2):
    """Responde com Chain-of-Thought: cita trechos curtos, numera o raciocínio e dá a
    resposta final após o marcador RESPOSTA: (mesmo padrão de c02 §1.4).

    Se o marcador faltar (por exemplo, saída cortada por falta de tokens), reenvia o
    problema ao modelo uma vez pedindo uma versão mais curta — mesmo tratamento de erro
    de gerar_json em c02. Devolve (saida_completa, resposta_final_ou_None)."""
    system = (
        "You are an assistant that answers strictly based on the following excerpts from "
        "class transcripts. If the excerpts do not contain the answer, say so explicitly "
        f"instead of guessing.\n<excerpts>\n{montar_contexto(resultados_busca)}\n</excerpts>"
    )
    user = (
        f"Question: {consulta}\n\n"
        "Think step by step: quote at most two SHORT fragments from the excerpts (a few "
        "words each, never a full excerpt), then number your reasoning steps (at most 3, "
        "one short sentence each), and only then give the final answer on a new line "
        f"starting with '{MARCADOR_RESPOSTA}'. The final answer must have at most 3 "
        "sentences. IMPORTANT: write ALL your reasoning steps and the final answer in "
        "Portuguese, not in English."
    )
    prompt_user = user
    for tentativa in range(1, max_tentativas + 1):
        saida = gerar(system, prompt_user, max_new_tokens=1024)
        m = re.search(rf"{re.escape(MARCADOR_RESPOSTA)}\s*(.+)", saida, re.DOTALL)
        if m:
            return saida, m.group(1).strip()
        if tentativa < max_tentativas:
            exibir_aviso(f"⚠️ Tentativa {tentativa}: marcador '{MARCADOR_RESPOSTA}' ausente — reenviando o problema ao modelo...")
            prompt_user = (
                f"{user}\n\nYour previous answer was cut off or did not include the line "
                f"starting with '{MARCADOR_RESPOSTA}'. Do it again more briefly: shorter "
                "quotes, at most 3 short reasoning steps, and the final answer after "
                f"'{MARCADOR_RESPOSTA}'."
            )
    exibir_aviso(f"❌ Saída não conforme após {max_tentativas} tentativas: marcador '{MARCADOR_RESPOSTA}' ausente.", tipo="erro")
    return saida, None


resultado_1 = next(r for r in RESULTADOS if r["id"] == 1)
resultado_3 = next(r for r in RESULTADOS if r["id"] == 3)

saida_boa, resposta_boa = responder_com_contexto(resultado_1["consulta"], resultado_1["bruta"])
saida_ruim, resposta_ruim = responder_com_contexto(resultado_3["consulta"], resultado_3["bruta"])

def exibir_resposta_cot(rotulo, consulta, saida, resposta_final):
    """Apresenta cada caso como um bloco único e coeso, em vez de caixas separadas: o
    raciocínio (o que vem antes do marcador) aparece atenuado, como etapa intermediária,
    e a resposta final aparece em destaque logo abaixo."""
    exibir_titulo(f'{rotulo}: "{consulta}"')
    corte = saida.rfind(MARCADOR_RESPOSTA) if resposta_final is not None else len(saida)
    raciocinio = saida[:corte].strip()

    def _como_html(texto):
        t = _html.escape(texto)
        t = re.sub(r"^#{1,6}\s+(.+)$", r"<strong>\1</strong>", t, flags=re.MULTILINE)
        t = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", t)
        return t.replace("\n", "<br>")

    bloco_resposta = (
        f'<div style="font-weight:600;">{_como_html(resposta_final)}</div>'
        if resposta_final is not None else
        '<div style="color:#c0392b;">Resposta final não extraída — o marcador não veio '
        'mesmo após a retentativa; o raciocínio acima é tudo o que o modelo devolveu.</div>'
    )
    display(HTML(
        '<div style="background:#eafaf0;border-left:4px solid #2f9e5c;border-radius:4px;'
        'padding:10px 14px;margin:0 0 12px 0;font-family:system-ui,-apple-system,sans-serif;'
        'line-height:1.55;color:#111827;">'
        '<div style="color:#6b7280;font-size:0.9em;font-weight:600;margin-bottom:4px;">'
        'Raciocínio — trechos citados e passos numerados</div>'
        f'<div style="color:#4b5563;font-style:italic;">{_como_html(raciocinio)}</div>'
        '<hr style="border:none;border-top:1px solid #bfe3cd;margin:10px 0;">'
        '<div style="color:#6b7280;font-size:0.9em;font-weight:600;margin-bottom:4px;">'
        f'Resposta final</div>{bloco_resposta}'
        '</div>'
    ))


exibir_resposta_cot("Consulta com recuperação BOA",
                    f'{resultado_1["consulta_es"]} ({resultado_1["consulta_pt"]})',
                    saida_boa, resposta_boa)
exibir_resposta_cot("Consulta com recuperação RUIM (fora do corpus)",
                    f'{resultado_3["consulta_es"]} ({resultado_3["consulta_pt"]})',
                    saida_ruim, resposta_ruim)

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**📋 O que os resultados demonstram, com `gemini-3.5-flash`**

Os resultados confirmam a expectativa nos dois sentidos, já com o modelo mais barato.

**Recuperação boa vira resposta boa.** Na consulta 1, a resposta cita fatos concretos do
contexto recuperado — LangChain, Hugging Face, Chroma, Llama e Ollama, além das APIs da
OpenAI e da Gemini e da ferramenta local GPT4All — gerada com `gemini-3.5-flash`, o modelo
escolhido por custo e eficiência.

**Recuperação ruim não vira resposta inventada.** Na consulta 3, fora do corpus, o modelo
não tentou adivinhar: respondeu que as transcrições não contêm informações sobre o
tratamento de doença renal crônica em gatos. A instrução explícita de não inventar
funcionou, mas ela não age sozinha: foi a §4 que, pelo score baixo, já havia sinalizado
que a recuperação era ruim. Sem esse sinal prévio, não haveria como distinguir uma
resposta confiável de uma resposta alucinada apresentada com confiança.

**O que o Chain-of-Thought acrescenta.** Com o raciocínio visível — os trechos citados e
os passos numerados antes do marcador `RESPOSTA:` — dá para conferir se a resposta final
veio mesmo das linhas recuperadas, e não de conhecimento externo do modelo.

**Custo.** Todas as gerações deste notebook usam `gemini-3.5-flash`, escolhido por custo e
eficiência conforme a justificativa da introdução, sem perda de qualidade perceptível
nessas duas tarefas curtas.

</div>

---

## §6 Conclusão

<div style="background:#eef6ff;border-left:4px solid #2b7de9;padding:10px 14px;border-radius:4px;color:#111827;">

**📋 Quando a recuperação funcionou, quando falhou, e o que isso fez com a resposta final**

O caminho deste notebook foi: cortar as transcrições em trechos com sobreposição,
transformar cada trecho em um vetor com o embedding do Gemini, guardar tudo em um índice
FAISS e testar a busca com quatro consultas pensadas para forçar situações diferentes.
Esta conclusão discute o que esses testes mostraram, em três perguntas: quando a
recuperação funcionou bem, quando falhou, e como isso afetou a qualidade da resposta
final.

**Quando a recuperação funcionou bem.** Em três das quatro consultas, com scores na faixa
aproximada de 0,65 a 0,74: a consulta sobre as ferramentas do curso trouxe o trecho certo
com a lista de bibliotecas; a consulta sobre os erros de transcrição encontrou o assunto
correto mesmo com o transcript escrevendo "Ruginface" em vez de Hugging Face — a busca por
significado tolerou o erro de grafia; e a consulta sobre logística trouxe trechos de duas
aulas diferentes. A expansão de consulta (HyDE) ajudou nesses casos bons: reescrever a
pergunta como se fosse uma fala de aula subiu os scores (de cerca de 0,72 para 0,74 numa,
de 0,69 para 0,71 na outra), com dois poréns — o trecho devolvido ficou mais estreito, e a
reescrita feita pelo `gemini-3.5-flash` cobriu o assunto certo mas não repetiu a grafia
errada "Ruginface" do transcript.

<div style="background:#eafaf0;border-left:4px solid #2f9e5c;border-radius:4px;padding:6px 12px;margin:8px 0;font-size:0.95em;">
✅ <strong>Exemplo:</strong> pergunta <code>"Hugging Face and CUDA"</code> → trecho
recuperado: <em>"a gente vai como ferramental trabalhar com
<strong>ruginface</strong>, com Api dessas"</em> (score ≈ 0,68) — a grafia errada do
transcript não impediu a busca de achar o assunto certo.
</div>

**Quando a recuperação falhou.** Na consulta feita de propósito sobre um assunto que não
existe nas aulas (doença renal em gatos), o score caiu para cerca de 0,53, bem abaixo da
faixa das consultas boas. E essa queda é exatamente o que queríamos ver: como o índice
sempre devolve os k vizinhos mais próximos, ele nunca responde "não achei" — o score baixo
é o único sinal de que não há resposta boa no corpus, em vez de o sistema entregar um
resultado ruim com confiança alta.

<div style="background:#fff8e6;border-left:4px solid #d9a404;border-radius:4px;padding:6px 12px;margin:8px 0;font-size:0.95em;">
⚠️ <strong>Exemplo:</strong> pergunta <code>"What is the recommended treatment for
chronic kidney disease in cats?"</code> → o melhor resultado foi um trecho de aula sem
relação com o tema, com score ≈ 0,53 — bem abaixo da faixa de 0,65 a 0,74 das
consultas boas.
</div>

**Como isso afetou a resposta final.** A qualidade da resposta ficou limitada pela
qualidade do que foi recuperado, nos dois sentidos. Com recuperação boa, a resposta citou
fatos concretos dos trechos: LangChain, Hugging Face, Chroma, Llama, Ollama e as APIs da
OpenAI e do Gemini. Com recuperação ruim, a resposta não virou invenção: o modelo admitiu
que as transcrições não continham a informação, em vez de inventar um tratamento para a
pergunta sobre gatos — e isso só foi possível porque o score baixo já tinha sinalizado a
falha antes; sem esse sinal, não haveria como distinguir uma resposta confiável de uma
alucinação apresentada com confiança.
Três detalhes do desenho ajudam a explicar esse resultado. Primeiro, a resposta só
enxerga os três trechos mais próximos que a busca devolve: se o trecho certo não entra
nesse top-3, não há modelo bom que conserte — o que foi recuperado é o teto da
resposta. Segundo, a recusa a inventar não foi mérito espontâneo do modelo: o prompt
manda responder somente com base nos trechos fornecidos e dizer explicitamente quando
eles não contêm a resposta. Terceiro, como a resposta é gerada com Chain-of-Thought —
o modelo cita os trechos e numera os passos antes da linha `RESPOSTA:` — dá para
conferir que cada afirmação saiu mesmo das linhas recuperadas, e não do conhecimento
prévio do modelo.

Quanto ao custo, todas as gerações do notebook — a reescrita das consultas, os resumos por
trecho nas tabelas e a resposta final — usaram o `gemini-3.5-flash`, escolhido por custo e
eficiência conforme a justificativa da introdução, sem perda de qualidade perceptível
nessas tarefas curtas.

**O índice e os chunks ficam salvos em disco.** Ao final da indexação, o índice FAISS é gravado em `data/processed/indice_faiss_c03.index` e os metadados dos chunks — junto com a configuração do modelo de embeddings (modelo, dimensão e task types) — em `data/processed/chunks_c03.json`. Assim, o pipeline RAG de c05 **carrega esses artefatos prontos** em vez de reindexar o corpus: os embeddings já foram pagos uma vez aqui, e a recuperação de c05 roda sobre exatamente o mesmo índice que este notebook avaliou.

**Implicações para o pipeline RAG completo, planejado para c05**: a estratégia de
chunking, tamanho e sobreposição, e o uso de `task_type` assimétrico não são detalhes de
implementação; são decisões que determinam se a recuperação vai funcionar. Testar
consultas adversariais, fora do corpus, antes de confiar no sistema é o que evita que uma
recuperação ruim se transforme em uma resposta alucinada com aparência de confiança. Do
lado do custo, vale a regra adotada aqui: reservar o modelo caro para tarefas em que já
foi medida uma diferença real de qualidade, e usar o modelo barato para as demais.

</div>